# 03. Layer 3: dự báo lỗi thời

TabPFN v2 (teacher, cần GPU cho nhanh) sinh soft-label out-of-fold, XGBoost (student) học lại. Chọn α trên validation, báo cáo trên test theo thời gian.

In [1]:
import os, sys, subprocess
from importlib import metadata
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "https://github.com/vinhlee9x/conflict-detection-obsolescence-forecasting.git", "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
    core = ("numpy", "pandas", "scipy", "scikit-learn", "pyarrow")
    def installed_versions():
        found = {}
        for name in core:
            try:
                found[name] = metadata.version(name)
            except metadata.PackageNotFoundError:
                found[name] = None
        return found
    before = installed_versions()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    if installed_versions() != before:
        print("Đã cài phiên bản mới của numpy/pandas/scipy. Runtime tự khởi động lại, đợi vài giây rồi chạy lại notebook từ đầu.")
        os.kill(os.getpid(), 9)
else:
    while not os.path.exists("src") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Thư mục làm việc:", os.getcwd())

Thư mục làm việc: /content/repo


In [2]:
# Tuỳ chọn: giữ data/ và artifacts/ trên Google Drive giữa các session Colab
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/conflict-detection"
    for folder in ("data/processed", "artifacts"):
        os.makedirs(f"{target}/{folder}", exist_ok=True)
        subprocess.run(f"cp -rn {folder}/. {target}/{folder}/ && rm -rf {folder} && ln -s {target}/{folder} {folder}", shell=True, check=True)

Mounted at /content/drive


In [3]:
from src import pipeline, settings
import pandas as pd
settings.TABPFN_MAX_CONTEXT = 3000
summary = pipeline.layer3("kb", teacher="tabpfn")
summary["rows"], summary["positives"], summary["best_alpha"]

({'train': 5623, 'valid': 838, 'test': 1046},
 {'train': 357, 'valid': 45, 'test': 52},
 0.7)

In [4]:
pd.DataFrame(summary["test"])

,model,roc_auc,pr_auc,precision_at_10,precision_at_20,positives,n,ms_per_row
0,xgb_hard_labels,0.783451,0.247201,0.7,0.4,52,1046,0.004668
1,xgb_distilled,0.777918,0.266307,0.8,0.5,52,1046,0.004774
2,tabpfn,0.787649,0.309069,1.0,0.5,52,1046,4.711496
